# NeuroObfuscator v6 — QLoRA fine-tuning (T4 GPU)

Обучение модели, генерирующей **план обфускации** (JSON, без seed) по JS-коду + AST-признакам.

**Датасет v6**: `data/final_v6/{train,val}.jsonl` — 6 000 train / 750 val (загрузить `final_v6.zip` в Google Drive).

**Что нового в v6:**
- operator_sub adoption 45% -> 99.7% (relevance bonus в скоринге)
- string_encode adoption 83% -> 98.3%
- 3 новых строковых генератора (i18n, log, url)
- Контрастные intensity-sweep планы (--intensity-seeded)

## Железо и пресеты

| Пресет | GPU | Модель | Batch |
|---|---|---|---|
| `t4_3b` (основной на T4) | T4 15 GB | Qwen2.5-Coder-3B-Instruct, 4-bit | 8 x 2 = 16 |
| `l4_7b` (если L4/A100) | L4 22.5 GB | CodeLlama-7B-Instruct, 4-bit NF4 | 8 x 2 = 16 |

Пресет выбирается **автоматически** по VRAM (порог 20 GB), можно задать вручную (`MANUAL_PRESET`).

Гиперпараметры: r=32, alpha=64, lr=2e-4, 3 эпохи, cosine, effective batch 16.

**Время на T4 (3B)**: ~40-70 мин на 3 эпохи.


In [ ]:
%pip install unsloth
# Если Colab предложит перезапустить runtime — перезапустить и продолжить отсюда


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/neuroobfuscator/final_v6'  # поправить под свой путь в Drive
ADAPTER_OUT = '/content/drive/MyDrive/neuroobfuscator/adapters_v6'
!ls {DATA_DIR}


In [ ]:
import torch

# None = автодетект по VRAM; либо явно 't4_3b' / 'l4_qwen7b' / 'l4_codellama7b'
MANUAL_PRESET = None

PRESETS = {
    # T4 15 GB и слабее: Qwen2.5-Coder-3B-Instruct 4-bit
    't4_3b': {
        'model_name': 'unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit',
        'max_seq_len': 2048,
        'batch_size': 8,
        'grad_accum': 2,
    },
    # L4 22.5 GB: Qwen2.5-Coder-7B-Instruct 4-bit (рекомендуемый на L4)
    'l4_qwen7b': {
        'model_name': 'unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit',
        'max_seq_len': 2048,
        'batch_size': 8,
        'grad_accum': 2,
    },
    # L4 22.5 GB: CodeLlama-7B-Instruct 4-bit (v5-совместимый промпт [INST])
    'l4_codellama7b': {
        'model_name': 'unsloth/codellama-7b-instruct-bnb-4bit',
        'max_seq_len': 2048,
        'batch_size': 8,
        'grad_accum': 2,
    },
}

if MANUAL_PRESET:
    PRESET_NAME = MANUAL_PRESET
else:
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    gpu_name = torch.cuda.get_device_name(0)
    if vram_gb >= 20:
        PRESET_NAME = 'l4_qwen7b'   # L4/A100: 7B Qwen
    else:
        PRESET_NAME = 't4_3b'       # T4: 3B Qwen
    print(f'GPU: {gpu_name} ({vram_gb:.1f} GB) -> preset {PRESET_NAME}')

CFG = PRESETS[PRESET_NAME]
MODEL_NAME = CFG['model_name']
MAX_SEQ_LEN = CFG['max_seq_len']
IS_QWEN = 'qwen' in MODEL_NAME.lower()
print('model:', MODEL_NAME, '| bs:', CFG['batch_size'], 'x ga:', CFG['grad_accum'], '| qwen:', IS_QWEN)


In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CFG['model_name'],
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=64,
    lora_dropout=0.0,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=3407,
)
print('LoRA ready')


In [ ]:
# Датасет: keys = {id, instruction, output, metadata}
# instruction = [INST]-промпт (CodeLlama) или ChatML (Qwen) — берём как есть из jsonl
# output = чистый JSON без seed
import json
from datasets import Dataset

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return Dataset.from_list([json.loads(l) for l in f if l.strip()])

train_ds = load_jsonl(f'{DATA_DIR}/train.jsonl')
val_ds = load_jsonl(f'{DATA_DIR}/val.jsonl')
print('train:', len(train_ds), '| val:', len(val_ds))

EOS = tokenizer.eos_token

def format_example(rec):
    return {'text': rec['instruction'] + rec['output'] + EOS}

train_fmt = train_ds.map(format_example)
val_fmt = val_ds.map(format_example)
print(repr(train_fmt[0]['text'][-160:]))


In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_fmt,
    eval_dataset=val_fmt,
    args=SFTConfig(
        dataset_text_field='text',
        per_device_train_batch_size=CFG['batch_size'],
        gradient_accumulation_steps=CFG['grad_accum'],
        num_train_epochs=3,
        learning_rate=2e-4,
        lr_scheduler_type='cosine',
        warmup_ratio=0.03,
        logging_steps=10,
        eval_strategy='steps',
        eval_steps=100,
        save_strategy='steps',
        save_steps=200,
        save_total_limit=2,
        per_device_eval_batch_size=16,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        optim='adamw_8bit',
        weight_decay=0.01,
        seed=3407,
        output_dir='/content/checkpoints',
        report_to='none',
        max_seq_length=MAX_SEQ_LEN,
    ),
)


In [ ]:
# Маскирование промпта: loss только на JSON-комплишене.
# Определяем маркер конца промпта по модели.
import json
from transformers import DataCollatorForSeq2Seq

RESPONSE_MARKER = '<|im_start|>assistant' if IS_QWEN else '[/INST]'

def tokenize_with_mask(rec):
    full = rec['text']
    enc = tokenizer(
        full,
        truncation=True,
        max_length=MAX_SEQ_LEN,
        return_offsets_mapping=True,
    )
    idx = full.rfind(RESPONSE_MARKER)
    resp_start = (idx + len(RESPONSE_MARKER)) if idx != -1 else len(full)
    input_ids = enc['input_ids']
    labels = [
        (tid if b > resp_start else -100)
        for tid, (a, b) in zip(input_ids, enc['offset_mapping'])
    ]
    return {'input_ids': input_ids, 'labels': labels}

train_tok = train_fmt.map(
    tokenize_with_mask,
    remove_columns=train_fmt.column_names,
    desc='Tokenize + mask prompt',
)
val_tok = val_fmt.map(
    tokenize_with_mask,
    remove_columns=val_fmt.column_names,
    desc='Tokenize + mask prompt',
)

assert all(any(l != -100 for l in ex['labels']) for ex in train_tok.select(range(64)))

trainer.train_dataset = train_tok
trainer.eval_dataset = val_tok.select(range(min(200, len(val_tok))))
trainer.data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=-100,
)
print('prompt-masked datasets ready:', len(train_tok), 'train,', len(trainer.eval_dataset), 'eval-subset')


In [ ]:
trainer_stats = trainer.train()
print(f'train runtime: {trainer_stats.metrics.get("train_runtime", 0)/60:.1f} min')
print(f'train loss:    {trainer_stats.metrics.get("train_loss", 0):.4f}')


In [ ]:
# Быстрая проверка: генерация на валидационном примере (greedy)
FastLanguageModel.for_inference(model)

sample = val_ds[0]
inputs = tokenizer(sample['instruction'], return_tensors='pt').to('cuda')
out = model.generate(**inputs, max_new_tokens=256, temperature=0.1, do_sample=False)
generated = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print('EXPECTED:', sample['output'])
print('GENERATED:', generated)


In [ ]:
# Метрика Phase 3: JSON parse rate на 64 val-примерах (порог >= 95%)
import json

def extract_json(s):
    a, b = s.find('{'), s.rfind('}')
    if a == -1 or b <= a:
        return None
    try:
        return json.loads(s[a:b + 1])
    except Exception:
        return None

def validate_plan_schema(plan):
    ORDER = ["rename", "string_encode", "operator_sub", "dead_code", "opaque_predicates"]
    if not isinstance(plan, dict): return False
    if not all(k in plan for k in ['intensity', 'transforms', 'order']): return False
    if plan['intensity'] not in {'light', 'medium', 'heavy'}: return False
    if set(plan.get('transforms', {})) != set(ORDER): return False
    enabled = [n for n in ORDER if plan['transforms'][n].get('enabled')]
    return plan.get('order') == enabled

ok = schema_ok = total = 0
for i in range(min(64, len(val_ds))):
    rec = val_ds[i]
    inputs = tokenizer(rec['instruction'], return_tensors='pt').to('cuda')
    out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
    gen = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    total += 1
    plan = extract_json(gen)
    if plan is not None:
        ok += 1
        if validate_plan_schema(plan):
            schema_ok += 1

print(f'JSON parse rate:   {ok}/{total} = {ok/total:.1%} (target >= 95%)')
print(f'Schema valid rate: {schema_ok}/{total} = {schema_ok/total:.1%} (target >= 90%)')


In [ ]:
# Сохранить LoRA-адаптер на Drive (для inference.py)
model.save_pretrained(ADAPTER_OUT)
tokenizer.save_pretrained(ADAPTER_OUT)
print('saved to', ADAPTER_OUT)


In [ ]:
# Финальная проверка: JSON parse + schema rate на 200 val-примерах
# Порог: JSON >= 95%, schema >= 90%
import json

def extract_json(s):
    a, b = s.find('{'), s.rfind('}')
    if a == -1 or b <= a:
        return None
    try:
        return json.loads(s[a:b + 1])
    except Exception:
        return None

def validate_plan_schema(plan):
    ORDER = ["rename", "string_encode", "operator_sub", "dead_code", "opaque_predicates"]
    if not isinstance(plan, dict): return False
    if not all(k in plan for k in ['intensity', 'transforms', 'order']): return False
    if plan['intensity'] not in {'light', 'medium', 'heavy'}: return False
    if set(plan.get('transforms', {})) != set(ORDER): return False
    enabled = [n for n in ORDER if plan['transforms'][n].get('enabled')]
    return plan.get('order') == enabled

N = min(200, len(val_ds))
ok = schema_ok = total = 0
failures = []
for i in range(N):
    rec = val_ds[i]
    inputs = tokenizer(rec['instruction'], return_tensors='pt').to('cuda')
    out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
    gen = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    total += 1
    plan = extract_json(gen)
    if plan is None:
        failures.append(('json', i, gen[:120]))
        continue
    ok += 1
    if not validate_plan_schema(plan):
        failures.append(('schema', i, json.dumps(plan)[:120]))
        continue
    schema_ok += 1

print(f'JSON parse rate:   {ok}/{total} = {ok/total:.1%} (target >= 95%)')
print(f'Schema valid rate: {schema_ok}/{total} = {schema_ok/total:.1%} (target >= 90%)')
if failures:
    print(f'\nFailures ({len(failures)}):')
    for kind, i, snippet in failures[:5]:
        print(f'  [{kind}] val_ds[{i}]: {snippet}')

report = {
    'model': MODEL_NAME,
    'n_eval': total,
    'json_rate': ok / total,
    'schema_rate': schema_ok / total,
}
with open('/content/drive/MyDrive/neuroobfuscator/eval_v6_report.json', 'w') as f:
    json.dump(report, f, indent=2)
print('\nreport saved: eval_v6_report.json')


In [ ]:
# Экспорт в GGUF (для llama.cpp / Ollama / LM Studio)
# q4_k_m: ~5 GB для 7B, ~2 GB для 3B — лучший баланс размер/качество
# альтернатива: 'q8_0' (~8.5 GB для 7B, почти без потерь качества)

GGUF_OUT = '/content/drive/MyDrive/neuroobfuscator/gguf_v6'

model.save_pretrained_gguf(
    GGUF_OUT,
    tokenizer,
    quantization_method='q4_k_m',
)
print('GGUF saved to', GGUF_OUT)
!ls -la {GGUF_OUT}
